In [1]:
import xarray as xr
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.metrics import r2_score


In [2]:
fishing_ds = xr.open_dataset("./data/processed/dynamic/presence_ESP_TRAWL.nc")
fishing = fishing_ds["presence"]
fishing = fishing.fillna(0)

mask_ds = xr.open_dataset("./data/processed/static/fishing_area_mask.nc")
mask = mask_ds["mask"]

temp_ds = xr.open_dataset("./data/processed/dynamic/to_surface.nc")
temp_ds = temp_ds.reset_coords("depth", drop=True)
temp = temp_ds["to"]
temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_bottom_ds = xr.open_dataset("./data/processed/dynamic/temp_bottom.nc")
temp_bottom = temp_bottom_ds["to"]
temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

chl_ds = xr.open_dataset("./data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

mixed_ds = xr.open_dataset("./data/processed/dynamic/mixed_layer.nc")
mixed = mixed_ds["mlotst"]
mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

depth_ds = xr.open_dataset("./data/processed/static/depth.nc")
depth = depth_ds["depth"] 
depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)

zo_ds = xr.open_dataset("./data/processed/dynamic/zo_surface.nc")
zo = zo_ds["zo"]
zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)

so_ds = xr.open_dataset("./data/processed/dynamic/so_surface.nc")
so = so_ds["so"]
so = (so - so.mean()) / so.std()
so = so.fillna(0)


month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)

lat = (temp["lat"] - temp["lat"].mean()) / temp["lat"].std()
lon = (temp["lon"] - temp["lon"].mean()) / temp["lon"].std()
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp) #se añade como dinamica porque ya se ha corregido la forma

temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so = xr.align(temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, join="inner")


#chanels
svars = 1
dvars = 10
time_steps = 6
in_channels= dvars+svars


In [3]:
class FishingDataset(Dataset):
    def __init__(self, fishing, temp, temp_bottom, chl, mixed, mask, depth, month_sin, month_cos, lat, lon, zo, so, time_steps=time_steps):
        # dynamic
        self.time_steps = time_steps
        self.fishing = fishing
        self.temp = temp
        self.chl = chl
        self.temp_bottom = temp_bottom
        self.mixed = mixed
        self.month_sin = month_sin
        self.month_cos = month_cos
        self.lat = lat
        self.lon = lon
        self.zo = zo
        self.so = so
        # static
        self.mask = mask.values.astype(np.float32)  # shape (H, W)
        self.depth = depth.values.astype(np.float32)  # shape (H, W)
        self.times = fishing.time.values



    def __len__(self):
        return len(self.times) - self.time_steps

    def __getitem__(self, idx):
        t0 = idx
        t1 = idx + self.time_steps

        # dynamic sequences: (time_steps, H, W)
        temp_seq = self.temp.isel(time=slice(t0, t1)).values.astype(np.float32)
        chl_seq  = self.chl.isel(time=slice(t0, t1)).values.astype(np.float32)
        temp_bottom_seq  = self.temp_bottom.isel(time=slice(t0, t1)).values.astype(np.float32)
        mixed_seq = self.mixed.isel(time=slice(t0, t1)).values.astype(np.float32)
        month_sin_seq = self.month_sin.isel(time=slice(t0, t1)).values.astype(np.float32)
        month_cos_seq = self.month_cos.isel(time=slice(t0, t1)).values.astype(np.float32)
        lat_seq = self.lat.isel(time=slice(t0, t1)).values.astype(np.float32)
        lon_seq = self.lon.isel(time=slice(t0, t1)).values.astype(np.float32)
        zo_seq = self.zo.isel(time=slice(t0, t1)).values.astype(np.float32)
        so_seq = self.so.isel(time=slice(t0, t1)).values.astype(np.float32)

        # stack dynamic channels
        x_dyn = np.stack([temp_seq, chl_seq, temp_bottom_seq, mixed_seq, month_cos_seq, month_sin_seq, lat_seq, lon_seq, zo_seq, so_seq], axis=1)

        #static chanels
        depth_ch = self.depth[np.newaxis, np.newaxis, ...]  # (1,1,H,W)
        depth_ch = np.repeat(depth_ch, self.time_steps, axis=0)  # (T,1,H,W)


        x = np.concatenate([x_dyn, depth_ch], axis=1)  # (T, C+1, H, W)

        # target and mask -> (1, H, W)
        y = self.fishing.isel(time=t1).values.astype(np.float32)[np.newaxis, ...]
        m = self.mask[np.newaxis, ...]
        
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(m)



dataset = FishingDataset(fishing, temp, temp_bottom, chl, mixed, mask, depth, month_sin, month_cos, lat, lon,  zo, so, time_steps=time_steps)
n = len(dataset)

total_sequences = len(dataset)
train_end = int(0.7 * total_sequences)  # First 70% of time
val_end = train_end + int(0.15 * total_sequences)  # Next 15%

train_dataset = torch.utils.data.Subset(dataset, range(0, train_end))
val_dataset = torch.utils.data.Subset(dataset, range(train_end, val_end))
test_dataset = torch.utils.data.Subset(dataset, range(val_end, total_sequences))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [4]:
# -------- Conv Block --------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm3d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv3d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm3d(out_ch),
                nn.ReLU(inplace=True)
            )

    def forward(self, x):
        return self.block(x)

# -------- U-Net 3D --------
class UNet3D(nn.Module):
    def __init__(self, in_channels, out_channels=1):
        super().__init__()
        # Encoder
        self.enc1 = ConvBlock(in_channels, 32)
        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool3d(2)

        # Bottleneck
        self.bottleneck = ConvBlock(64, 64)

        # Decoder
        self.up2 = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.dec2 = ConvBlock(32 + 64, 32) # 32 from up, 64 from skip = 96

        self.up1 = nn.ConvTranspose3d(32, 16, 2, stride=2)
        self.dec1 = ConvBlock(16 + 32, 32) # 16 from up, 32 from skip = 48

        self.final_conv = nn.Conv3d(32, out_channels, 1)

        ### How de channels affect the performance and accuracy of the model????
    def forward(self, x):
        x = x.permute(0, 2, 1, 3, 4) # (B, C, T, H, W) ### why i need to do this, why the channels are in the wrong order?

        s1 = self.enc1(x)
        p1 = self.pool1(s1)
        
        s2 = self.enc2(p1)
        p2 = self.pool2(s2)

        b = self.bottleneck(p2)

        d2 = self.up2(b)
        d2 = F.interpolate(d2, size=s2.shape[2:], mode='trilinear', align_corners=False)
        d2 = torch.cat([d2, s2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = F.interpolate(d1, size=s1.shape[2:], mode='trilinear', align_corners=False)
        d1 = torch.cat([d1, s1], dim=1)
        d1 = self.dec1(d1)

        out = self.final_conv(d1)
        return out[:, :, -1, :, :] # Return only the last time step, this is because of the mask only uses 1 time step

In [5]:
# class CNN3D(nn.Module):
#     def __init__(self, in_channels):
#         super().__init__()

#         # -------- Feature branch --------
#         self.feature = nn.Sequential(
#             nn.Conv3d(in_channels, 32, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool3d(2),
#             nn.Conv3d(32, 64, 3, padding=1),
#             nn.ReLU(),
#             nn.MaxPool3d(2)
#         )

#         # -------- Attention branch --------
#         self.gap = nn.AdaptiveAvgPool3d(1)  # (B, C, 1,1,1)

#         self.fc1 = nn.Conv3d(in_channels, in_channels // 2, 1)
#         self.fc2 = nn.Conv3d(in_channels // 2, 64, 1)


#         # -------- Fusion and output --------
#         self.conv_out = nn.Sequential(
#             nn.Conv3d(64, 128, 3, padding=1, groups=1),
#             nn.BatchNorm3d(128),
#             nn.ReLU(),
#             nn.Conv3d(128, 1, 1)
#         )

#     def forward(self, x):
#         # (B, T, C, H, W) → (B, C, T, H, W)
#         x = x.permute(0, 2, 1, 3, 4)

#         # -------- Feature path --------
#         feat = self.feature(x)  # (B,64,T,H,W)

#         # -------- Attention path --------
#         attn = self.gap(x) # feat oror x??          # (B,C,1,1,1)
#         attn = F.relu(self.fc1(attn))   # (B,16,1,1,1)
#         attn = torch.sigmoid(self.fc2(attn))  # (B,64,1,1,1)

#         # -------- Fusion --------
#         feat = feat * attn 

#         # -------- Output --------
#         out = self.conv_out(feat)

#         out = out.mean(dim=2)
#         return out
    
def masked_mse_loss(pred, target, mask):
    mask = mask.unsqueeze(1)  # match channel dimension
    loss = (pred - target) ** 2
    loss = loss * mask
    return loss.sum() / mask.sum()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet3D(in_channels=in_channels)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) ### how to choose the learning rate?
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True) using something like this?


best_val_r2 = -float("inf")
best_train_r2 = -float("inf")
patience = 15
epochs =150
counter = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    train_preds, train_targets = [], []

    for x, y, m in train_loader:
        x = x.to(device)
        y = y.to(device)
        m = m.to(device)


        pred = model(x)

        loss = masked_mse_loss(pred, y, m)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

        mask_flat = m.detach().cpu().view(-1) > 0
        train_preds.append(pred.detach().cpu().view(-1)[mask_flat])
        train_targets.append(y.detach().cpu().view(-1)[mask_flat])
    
    # compute R2
    r2 = r2_score(torch.cat(train_targets).numpy(), torch.cat(train_preds).numpy())



    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for x, y, m in val_loader:
            x, y = x.to(device), y.to(device)
            m = m.to(device)
            pred = model(x)

            mask_flat = m.cpu().view(-1) > 0

            val_preds.append(pred.cpu().view(-1)[mask_flat])
            val_targets.append(y.cpu().view(-1)[mask_flat])

    val_r2 = r2_score(torch.cat(val_targets).numpy(), torch.cat(val_preds).numpy())

    if val_r2 > best_val_r2:
        counter = 0
        best_val_r2 = val_r2
    if r2 > best_train_r2:
        counter=0
        best_train_r2 = r2
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break

    print(f"Epoch {epoch}, Loss: {total_loss/len(train_loader)}, R2: {r2}, Val R2: {val_r2}")

Epoch 0, Loss: 0.41350908536463976, R2: -2.6832430362701416, Val R2: -0.130476713180542
Epoch 1, Loss: 0.10856181208044291, R2: 0.01885586977005005, Val R2: 0.017466306686401367
Epoch 2, Loss: 0.10673914782702923, R2: 0.03653520345687866, Val R2: 0.04060232639312744
Epoch 3, Loss: 0.10561880778521299, R2: 0.048460960388183594, Val R2: 0.05289280414581299
Epoch 4, Loss: 0.10489803593605757, R2: 0.056047022342681885, Val R2: 0.06299400329589844
Epoch 5, Loss: 0.10425477340817452, R2: 0.06293004751205444, Val R2: 0.07269066572189331
Epoch 6, Loss: 0.10385744709521533, R2: 0.06724965572357178, Val R2: 0.07944536209106445
Epoch 7, Loss: 0.10361838478595019, R2: 0.06980365514755249, Val R2: 0.08498537540435791
Epoch 8, Loss: 0.10351373270153999, R2: 0.07105350494384766, Val R2: 0.0896720290184021
Epoch 9, Loss: 0.1035024518147111, R2: 0.07168668508529663, Val R2: 0.09473311901092529
Epoch 10, Loss: 0.10315811071544885, R2: 0.07515060901641846, Val R2: 0.1006014347076416
Epoch 11, Loss: 0.102